In [47]:
%pip install typing

Note: you may need to restart the kernel to use updated packages.


In [48]:
from csp import Constraint, CSP
from typing import Dict, List, Optional, Callable
import random

## Boolean satisfiability problem (3-SAT)
### Constraint Satisfaction Problem

In logic and computer science, the Boolean satisfiability problem (sometimes called propositional satisfiability problem and abbreviated SATISFIABILITY, SAT or B-SAT) asks whether there exists an interpretation that satisfies a given Boolean formula. In other words, it asks whether the formula's variables can be consistently replaced by the values TRUE or FALSE to make the formula evaluate to TRUE. If this is the case, the formula is called satisfiable, else unsatisfiable. For example, the formula "a AND NOT b" is satisfiable because one can find the values a = TRUE and b = FALSE, which make (a AND NOT b) = TRUE. In contrast, "a AND NOT a" is unsatisfiable.

### A small note on the problem's relevance

SAT is the first problem that was proven to be NP-complete—this is the Cook–Levin theorem. This means that all problems in the complexity class NP, which includes a wide range of natural decision and optimization problems, are at most as difficult to solve as SAT. There is no known algorithm that efficiently solves each SAT problem (where "efficiently" means "deterministically in polynomial time"). Although such an algorithm is generally believed not to exist, this belief has not been proven or disproven mathematically. Resolving the question of whether SAT has a polynomial-time algorithm would settle the P versus NP problem - one of the most important open problems in the theory of computing.

## Input

In [49]:
input_file = 'csp.in'


with open(input_file, 'r', encoding='utf-8') as f:
        content = list(map(int, f.read().split('\n')))

In [50]:
random_seed = int(content[0])
n_of_variables = int(content[1])

random.seed(random_seed)

It has been shown that constraints/variable_number = 4.3 is a critical point in wich obtaining a solution becomes the hardest on average, while still being possible to solve in an average of 50% of the cases.
See: 
[(Mitchell and Saleman, 1992)](https://www.researchgate.net/profile/David-Mitchell-68/publication/2769087_Hard_and_Easy_Distributions_of_SAT_Problems/links/54185f8c0cf25ebee9881a1d/Hard-and-Easy-Distributions-of-SAT-Problems.pdf?origin=publication_detail&_tp=eyJjb250ZXh0Ijp7ImZpcnN0UGFnZSI6InB1YmxpY2F0aW9uIiwicGFnZSI6InB1YmxpY2F0aW9uRG93bmxvYWQiLCJwcmV2aW91c1BhZ2UiOiJwdWJsaWNhdGlvbiJ9fQ)

In [51]:
def get_n_of_constraints(n_variables):
    return round(n_variables * 4.3)

First we generate a function to create 3-Sat conditionals, theese are the following caracteristics:
- A single variable ($A$) or its negation ($\neg A$).
- A group of literals connected by OR ($\vee$). In 3-SAT, each group has exactly three of these.
- The entire string of clauses connected by AND ($\wedge$).

In [52]:
def three_sat_contidional(A:bool, B:bool, C:bool, list_not_operator: List[bool]) -> bool:
    # Apply restrictions mentioned above
    # Note: A, B and C will be assigned randomly
    
    A = A if list_not_operator[0] else not A
    B = B if list_not_operator[1] else not B
    C = C if list_not_operator[2] else not C
    
    return A or B or C

Now we define variables and domain to apply CSP

In [53]:
variables: List[int] = range(n_of_variables)
domain: List[bool] = [True, False]

possible_values:Dict[int, list[bool]] = {}

for variable in variables:
    possible_values[variable] = domain

Create Constraint Class with:
 - satisfied (goal_test)

 As we know that no statement involves more than 3 Variables, we can define the constraint with X, Y, Z, and a callback function with the logic statment between those.

In [54]:
class LogicStatementConstraint(Constraint[str, int]):
    def __init__(self, X: int, Y: int, Z: int, statement: Callable[[bool, bool, bool], bool]):
        super().__init__(variables)
        self.x = X
        self.y = Y
        self.z = Z
        self.statement = statement
        
        
        
    def satisfied(self, assignment: Dict[int, bool]) -> bool:
        if (self.x not in assignment) or (self.y not in assignment) or (self.z not in assignment):
            return True
        
        x_value = assignment[self.x]
        y_value = assignment[self.y]
        z_value = assignment[self.z]
        
        return self.statement(x_value, y_value, z_value)

Implement CSP class with CSP algorithm (Inside a function, so that we can do the CSP n times)

In [55]:
def solve_csp(seed: int, n_of_variables: int) -> str:
    random.seed(seed)
    
    csp: CSP[str, int] = CSP(variables, possible_values)
    n_of_constraints = get_n_of_constraints(n_of_variables)
    not_operator_lists = [ [random.choice([True, False]) for _ in range(3)] for _ in range(n_of_constraints) ]
    
    for not_operator_list in not_operator_lists:
        random_key_lists = [ random.choice(variables) for _ in range( 3 )]  # 3 literals at a time

        # Pass a callable that captures the negation mask so it can be evaluated later
        csp.add_constraint(LogicStatementConstraint(
            random_key_lists[0],
            random_key_lists[1], 
            random_key_lists[2], 
            lambda A, B, C, ops=not_operator_list: three_sat_contidional(A, B, C, ops)
            ))



    solution: Optional[Dict[str, int]] = csp.backtracking_search()
    
    if solution is None:
        out = "No solution found!\n"
    else:
        out = str(solution) + '\n'
        
    return out
        



## Output

As output we will have a possible solution, or the string 'no solutions found!' for every problem imput

In [56]:
if __name__ == "__main__":
    
    output_dir = './csp.out'
    out = ''

    seed = 0
    n_var = 0
    # Input pairs are part if a single list in content variable
    for idx, parameter in enumerate(content):
    
        if idx % 2 == 0:
            seed = parameter
        
        # Number of variavles
        if idx % 2 == 1:
            n_var = parameter
            
            out += solve_csp(seed, n_var)
    
    with open(output_dir, "w") as file:
        file.write(out)

    print(f"Content written to {output_dir}")
    print(out)

Content written to ./csp.out
{0: True, 1: True, 2: True, 3: True, 4: True}
No solution found!
{0: True, 1: False, 2: False, 3: False, 4: True}
No solution found!
No solution found!
{0: True, 1: True, 2: True, 3: False, 4: True}

